In [1]:
import pandas as pd
import numpy as np

In [2]:
df_path = "D:\\vanshmalanidata\\Documents\\GitHub\\Uni-Dash_Reborn\\Machine_Learning_Algo"
df = pd.read_csv(df_path + "\\evaluation_dataset_ground_truth.csv")


In [3]:
df.head()

,sender_email,sender_domain,subject,clean_text,label_source,label_topic,label_urgency,model_topic,model_urgency,ground_truth_topic,ground_truth_urgency,summary
0,recommendations@discover.pinterest.com,discover.pinterest.com,Form Design for 23DCS023,form design for 23dcs023 to view this content ...,External / Misc,NaN,NaN,General Information / Misc,Low,General Information / Misc,Low,Access the form design for module 23DCS023 via...
1,nainaparmar.dcs@charusat.ac.in,charusat.ac.in,Fwd: Start Up Accelerator Program(SAP) initiat...,fwd: start up accelerator program(sap) initiat...,Institutional Sender,NaN,NaN,Internship / Placement Opportunities,Medium,Internship / Placement Opportunities,Medium,Upcoming Start Up Conclave and Accelerator Pro...
2,onlinecourses@nptel.iitm.ac.in,nptel.iitm.ac.in,Programming in Java - Week 09 Feedback Form,programming in java - week 09 feedback form nan,External Academic Platform,NaN,NaN,General Information / Misc,NaN,General Information / Misc,NaN,NaN
3,netacademail@external.cisco.com,external.cisco.com,NetAcad Student Newsletter – April 2025,netacad student newsletter – april 2025 hello ...,External / Misc,NaN,NaN,General Information / Misc,NaN,General Information / Misc,NaN,NaN
4,no-reply@accounts.google.com,accounts.google.com,Security alert,security alert [image: google] a new sign-in o...,External / Misc,NaN,NaN,Administrative / Fees / Counselling,Medium,Administrative / Fees / Counselling,Medium,Google has detected a new sign-in on your acco...


In [4]:
df = df.drop(
    columns=["label_topic", "label_urgency", "model_topic", "model_urgency"],
    errors="ignore"
)

df.head()

,sender_email,sender_domain,subject,clean_text,label_source,ground_truth_topic,ground_truth_urgency,summary
0,recommendations@discover.pinterest.com,discover.pinterest.com,Form Design for 23DCS023,form design for 23dcs023 to view this content ...,External / Misc,General Information / Misc,Low,Access the form design for module 23DCS023 via...
1,nainaparmar.dcs@charusat.ac.in,charusat.ac.in,Fwd: Start Up Accelerator Program(SAP) initiat...,fwd: start up accelerator program(sap) initiat...,Institutional Sender,Internship / Placement Opportunities,Medium,Upcoming Start Up Conclave and Accelerator Pro...
2,onlinecourses@nptel.iitm.ac.in,nptel.iitm.ac.in,Programming in Java - Week 09 Feedback Form,programming in java - week 09 feedback form nan,External Academic Platform,General Information / Misc,NaN,NaN
3,netacademail@external.cisco.com,external.cisco.com,NetAcad Student Newsletter – April 2025,netacad student newsletter – april 2025 hello ...,External / Misc,General Information / Misc,NaN,NaN
4,no-reply@accounts.google.com,accounts.google.com,Security alert,security alert [image: google] a new sign-in o...,External / Misc,Administrative / Fees / Counselling,Medium,Google has detected a new sign-in on your acco...


In [6]:
df.head()

,sender_email,sender_domain,subject,clean_text,label_source,ground_truth_topic,ground_truth_urgency,summary
0,recommendations@discover.pinterest.com,discover.pinterest.com,Form Design for 23DCS023,form design for 23dcs023 to view this content ...,External / Misc,General Information / Misc,Low,Access the form design for module 23DCS023 via...
1,nainaparmar.dcs@charusat.ac.in,charusat.ac.in,Fwd: Start Up Accelerator Program(SAP) initiat...,fwd: start up accelerator program(sap) initiat...,Institutional Sender,Internship / Placement Opportunities,Medium,Upcoming Start Up Conclave and Accelerator Pro...
2,onlinecourses@nptel.iitm.ac.in,nptel.iitm.ac.in,Programming in Java - Week 09 Feedback Form,programming in java - week 09 feedback form nan,External Academic Platform,General Information / Misc,NaN,NaN
3,netacademail@external.cisco.com,external.cisco.com,NetAcad Student Newsletter – April 2025,netacad student newsletter – april 2025 hello ...,External / Misc,General Information / Misc,NaN,NaN
4,no-reply@accounts.google.com,accounts.google.com,Security alert,security alert [image: google] a new sign-in o...,External / Misc,Administrative / Fees / Counselling,Medium,Google has detected a new sign-in on your acco...


In [7]:



OLLAMA_MODELS = [
    "qwen2.5:7b-instruct-q4_k_m",
    "mistral:7b-instruct-q4_k_m",
    "gemma:7b-instruct-q4_k_m",
]

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

LEVEL2_LABELS = [
    "Timetable / Schedule Update",
    "Exam Notifications",
    "Assignment or Submission",
    "Certification / Courses",
    "Internship / Placement Opportunities",
    "Events / Hackathons",
    "Important Announcements",
    "Administrative / Fees / Counselling",
    "General Information / Misc",
]

URGENCY_LABELS = ["Critical", "High", "Medium", "Low", "None"]


In [8]:

SYSTEM_PROMPT = f"""
You are an academic email assistant for a university student.

You will receive:
- SOURCE TRUST LEVEL (who sent the email)
- EMAIL SUBJECT
- EMAIL BODY

Important rules:
- Emails may be forwarded by faculty on behalf of administrative offices.
- Do NOT infer authority based only on sender.
- Decide topic and urgency from language, enforcement cues, deadlines, and tone.
- Output must be strictly valid JSON.
- Do NOT add explanations outside JSON.

Allowed topic labels:
{LEVEL2_LABELS}

Urgency definitions:
- Critical: deadline today or tomorrow; penalties if missed
- High: deadline within a few days; important action required
- Medium: relevant academic info, no immediate deadline
- Low: optional events or learning opportunities
- None: newsletters, promotions, routine info

Return JSON in this exact format:
{{
  "summary": "<one-line student-facing summary>",
  "label_topic": "<one of the allowed topic labels>",
  "label_urgency": "<one of the urgency labels>"
}}
"""

In [15]:
# ----------------------------------------------------------
# LLM Call
# ----------------------------------------------------------
import json
import requests

def classify_email(label_source, subject, clean_text, model_name):
    prompt = f"""
SOURCE TRUST LEVEL: {label_source}
SUBJECT: {subject}

EMAIL CONTENT:
{clean_text}
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model_name,
            "prompt": SYSTEM_PROMPT + "\n" + prompt,
            "stream": False,
            "options": {
                "temperature": 0,
                "top_p": 0.1
            }
        },
        timeout=180
    )

    response.raise_for_status()
    raw_output = response.json().get("response", "").strip()

    # ---- PRINT RAW LLM OUTPUT ----
    print("\n===== LLM RAW OUTPUT =====")
    print(raw_output)
    print("==========================\n")

    try:
        parsed = json.loads(raw_output)

        # ---- PRINT PARSED OUTPUT ----
        print(">>> PARSED RESULT:")
        print(parsed)
        print()

        return parsed

    except json.JSONDecodeError:
        print("!!! JSON PARSE FAILED — FALLING BACK !!!\n")
        return {
            "summary": None,
            "label_topic": None,
            "label_urgency": None,
            "json_valid": False,
            "raw_output": raw_output
}

In [16]:
df.shape

(150, 8)

In [17]:
# ----------------------------------------------------------
# Run LLM inference directly on df (NO sampling)
# ----------------------------------------------------------
OUTPUT_DIR = r"D:\vanshmalanidata\Documents\GitHub\Uni-Dash_Reborn\Machine_Learning_Algo"
import os
OLLAMA_MODELS=["gemma:7b-instruct-q4_k_m"]
for model_name in OLLAMA_MODELS:
    print("\n" + "=" * 100)
    print(f"RUNNING MODEL: {model_name}")
    print("=" * 100 + "\n")

    # Work on a fresh copy per model
    model_df = df.copy()
    model_df["summary"] = None
    model_df["label_topic"] = None
    model_df["label_urgency"] = None
    model_df["json_valid"] = None

    for idx, row in model_df.iterrows():
        result = classify_email(
            label_source=row["label_source"],
            subject=row.get("subject", ""),
            clean_text=row.get("clean_text", ""),
            model_name=model_name
        )

        model_df.at[idx, "summary"] = result.get("summary")
        model_df.at[idx, "label_topic"] = result.get("label_topic")
        model_df.at[idx, "label_urgency"] = result.get("label_urgency")
        model_df.at[idx, "json_valid"] = result.get("json_valid")

        if idx % 25 == 0:
            print(f"[{model_name}] Processed {idx} / {len(model_df)} emails")

    # ----------------------------------------------------------
    # Save per-model output into EXISTING run directory
    # ----------------------------------------------------------

    safe_model_name = model_name.replace(":", "_").replace("-", "_")
    LLM_OUTPUT_FILE = os.path.join(
        OUTPUT_DIR,
        f"llm_labeled_{safe_model_name}.csv"
    )

    model_df.to_csv(LLM_OUTPUT_FILE, index=False, encoding="utf-8")
    print(f"\nSaved results for model '{model_name}' to:")
    print(LLM_OUTPUT_FILE)

    cont_model = input(
        f"\nFinished model '{model_name}'. Continue to next model? (y/n): "
    ).strip().lower()

    if cont_model != "y":
        break


RUNNING MODEL: gemma:7b-instruct-q4_k_m


===== LLM RAW OUTPUT =====
```python
import json

# Extract information from email content
source_trust_level = "External / Misc"
subject = "Form Design for 23DCS023"
email_body = "form design for 23dcs023 to view this content open the following url in your browser: <url> pinterest 651 brannan street, san francisco, ca, 94107 help center: <url> privacy policy: <url> terms & conditions: <url> unsubscribe: <url>"

# Analyze email content
label_topic = "Assignment or Submission"
label_urgency = "High"

# Create JSON output
json_output = {
    "summary": "Form design for 23DCS023 email with information about the form, help center, privacy policy, and terms & conditions.",
    "label_topic": label_topic,
    "label_urgency": label_urgency
}

# Print JSON output
print(json.dumps(json_output))
```

**Output:**

```json
{
  "summary": "Form design for 23DCS023 email with information about the form, help center, privacy policy, and terms & conditions."

In [6]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

gt = pd.read_csv("evaluation_dataset_ground_truth.csv", dtype=str)

In [7]:
gt.head()

,sender_email,sender_domain,subject,clean_text,label_source,label_topic,label_urgency,summary
0,recommendations@discover.pinterest.com,discover.pinterest.com,Form Design for 23DCS023,form design for 23dcs023 to view this content ...,External / Misc,General Information / Misc,Low,Access the form design for module 23DCS023 via...
1,nainaparmar.dcs@charusat.ac.in,charusat.ac.in,Fwd: Start Up Accelerator Program(SAP) initiat...,fwd: start up accelerator program(sap) initiat...,Institutional Sender,Internship / Placement Opportunities,Medium,Upcoming Start Up Conclave and Accelerator Pro...
2,onlinecourses@nptel.iitm.ac.in,nptel.iitm.ac.in,Programming in Java - Week 09 Feedback Form,programming in java - week 09 feedback form nan,External Academic Platform,General Information / Misc,NaN,NaN
3,netacademail@external.cisco.com,external.cisco.com,NetAcad Student Newsletter – April 2025,netacad student newsletter – april 2025 hello ...,External / Misc,General Information / Misc,NaN,NaN
4,no-reply@accounts.google.com,accounts.google.com,Security alert,security alert [image: google] a new sign-in o...,External / Misc,Administrative / Fees / Counselling,Medium,Google has detected a new sign-in on your acco...


In [8]:
#the function to evaluate the model output
def evaluate_model(model_path, model_name):
    print("\n" + "="*80)
    print(f"EVALUATING: {model_name}")
    print("="*80)

    model_df = pd.read_csv(model_path, dtype=str)
    

    for label in ["label_topic", "label_urgency"]:
        print(f"\n--- {label.upper()} ---")
        print("Unique ground truth:")
        print(ground_truth[label].unique())

        print("\nUnique predictions:")
        print(model_df[label].unique())

        y_true = ground_truth[label]
        y_pred = model_df[label]

        print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
        print("\nClassification Report (Macro):")
        print(classification_report(y_true, y_pred, digits=4))

        print("Confusion Matrix:")
        print(confusion_matrix(y_true, y_pred))

In [9]:
evaluate_model("llm_labeled_qwen2.5_7b_instruct_q4_k_m.csv", "Qwen 7B Q4")
evaluate_model("llm_labeled_mistral_7b_instruct_q4_k_m.csv", "Mistral 7B Q4")
evaluate_model("llm_labeled_gemma_7b_instruct_q4_k_m.csv", "Gemma 7B Q4")


EVALUATING: Qwen 7B Q4

--- LABEL_TOPIC ---
Unique ground truth:
['General Information / Misc' 'Internship / Placement Opportunities'
 'Administrative / Fees / Counselling' 'Events / Hackathons'
 'Exam Notifications' 'Assignment or Submission'
 'Timetable / Schedule Update' 'Certification / Courses']

Unique predictions:
['General Information / Misc' 'Internship / Placement Opportunities'
 'Assignment or Submission' 'Administrative / Fees / Counselling'
 'Events / Hackathons' 'Exam Notifications' 'Timetable / Schedule Update'
 'Certification / Courses' 'Important Announcements']
Accuracy: 0.5467

Classification Report (Macro):
                                      precision    recall  f1-score   support

 Administrative / Fees / Counselling     1.0000    1.0000    1.0000         3
            Assignment or Submission     0.7692    0.7692    0.7692        13
             Certification / Courses     0.2222    0.6667    0.3333         6
                 Events / Hackathons     0.5625    

d:\vanshmalanidata\Documents\GitHub\Uni-Dash_Reborn\Machine_Learning_Algo\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\vanshmalanidata\Documents\GitHub\Uni-Dash_Reborn\Machine_Learning_Algo\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\vanshmalanidata\Documents\GitHub\Uni-Dash_Reborn\Machine_Learning_Algo\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` par

TypeError: '<' not supported between instances of 'float' and 'str'